In [1]:
import pandas as pd

In [2]:
dataset = pd.read_csv("insurance_pre.csv")

In [3]:
dataset

,age,sex,bmi,children,smoker,charges
0,19,female,27.900,0,yes,16884.92400
1,18,male,33.770,1,no,1725.55230
2,28,male,33.000,3,no,4449.46200
3,33,male,22.705,0,no,21984.47061
4,32,male,28.880,0,no,3866.85520
...,...,...,...,...,...,...
1333,50,male,30.970,3,no,10600.54830
1334,18,female,31.920,0,no,2205.98080
1335,18,female,36.850,0,no,1629.83350
1336,21,female,25.800,0,no,2007.94500


In [4]:
dataset = pd.get_dummies(dataset,dtype=int,drop_first=True)

In [5]:
dataset

,age,bmi,children,charges,sex_male,smoker_yes
0,19,27.900,0,16884.92400,0,1
1,18,33.770,1,1725.55230,1,0
2,28,33.000,3,4449.46200,1,0
3,33,22.705,0,21984.47061,1,0
4,32,28.880,0,3866.85520,1,0
...,...,...,...,...,...,...
1333,50,30.970,3,10600.54830,1,0
1334,18,31.920,0,2205.98080,0,0
1335,18,36.850,0,1629.83350,0,0
1336,21,25.800,0,2007.94500,0,0


In [6]:
dataset.columns

Index(['age', 'bmi', 'children', 'charges', 'sex_male', 'smoker_yes'], dtype='str')

In [11]:
indep = dataset[['age', 'bmi', 'children', 'sex_male', 'smoker_yes']]

In [12]:
indep

,age,bmi,children,sex_male,smoker_yes
0,19,27.900,0,0,1
1,18,33.770,1,1,0
2,28,33.000,3,1,0
3,33,22.705,0,1,0
4,32,28.880,0,1,0
...,...,...,...,...,...
1333,50,30.970,3,1,0
1334,18,31.920,0,0,0
1335,18,36.850,0,0,0
1336,21,25.800,0,0,0


In [13]:
dep = dataset[['charges']]

In [14]:
dep

,charges
0,16884.92400
1,1725.55230
2,4449.46200
3,21984.47061
4,3866.85520
...,...
1333,10600.54830
1334,2205.98080
1335,1629.83350
1336,2007.94500


In [16]:
# split into training set and test set
from sklearn.model_selection import train_test_split
x_train,x_test,y_train,y_test = train_test_split(indep,dep,test_size=1/3,random_state=0)

In [20]:
from sklearn.preprocessing import StandardScaler
sc=StandardScaler()
x_train = sc.fit_transform(x_train)
x_test = sc.transform(x_test)

In [28]:
param_grid = {
    'criterion': ['squared_error', 'absolute_error', 'poisson'],
    'max_depth': [None, 3, 5, 7, 10],
    'min_samples_split': [2, 5, 10],
    'min_samples_leaf': [1, 2, 4]
}

In [29]:
from sklearn.model_selection import GridSearchCV
from sklearn.tree import DecisionTreeRegressor

dtree = DecisionTreeRegressor(random_state=42)

grid_search = GridSearchCV(
    estimator=dtree,
    param_grid=param_grid,
    cv=5,
    scoring='r2',
    n_jobs=-1,
    verbose=1
)

grid_search.fit(x_train, y_train)

print("Best Parameters:", grid_search.best_params_)
print("Best R2 Score:", grid_search.best_score_)

Fitting 5 folds for each of 135 candidates, totalling 675 fits
Best Parameters: {'criterion': 'absolute_error', 'max_depth': 5, 'min_samples_leaf': 4, 'min_samples_split': 10}
Best R2 Score: 0.8298257491757859


In [30]:
# print best parameter after tuning
re = grid_search.cv_results_
print ("r_score value best parameter {}:".format(grid_search.best_params_))

r_score value best parameter {'criterion': 'absolute_error', 'max_depth': 5, 'min_samples_leaf': 4, 'min_samples_split': 10}:


In [31]:
table = pd.DataFrame.from_dict(re)

In [32]:
table

,mean_fit_time,std_fit_time,mean_score_time,std_score_time,param_criterion,param_max_depth,param_min_samples_leaf,param_min_samples_split,params,split0_test_score,split1_test_score,split2_test_score,split3_test_score,split4_test_score,mean_test_score,std_test_score,rank_test_score
0,0.012518,0.001305,0.005592,0.000937,squared_error,None,1,2,"{'criterion': 'squared_error', 'max_depth': No...",0.668571,0.517580,0.797425,0.631397,0.651736,0.653342,0.089339,132
1,0.008933,0.001177,0.004370,0.001247,squared_error,None,1,5,"{'criterion': 'squared_error', 'max_depth': No...",0.721220,0.541272,0.775600,0.661642,0.701805,0.680308,0.078608,127
2,0.008476,0.001093,0.003996,0.001018,squared_error,None,1,10,"{'criterion': 'squared_error', 'max_depth': No...",0.745883,0.613770,0.806387,0.678664,0.727970,0.714535,0.064869,119
3,0.007929,0.000509,0.003426,0.000283,squared_error,None,2,2,"{'criterion': 'squared_error', 'max_depth': No...",0.744699,0.603275,0.778507,0.715941,0.734227,0.715330,0.059611,118
4,0.009888,0.002871,0.003815,0.001275,squared_error,None,2,5,"{'criterion': 'squared_error', 'max_depth': No...",0.762219,0.611048,0.782016,0.722004,0.746079,0.724673,0.060123,114
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
130,0.006315,0.000106,0.002891,0.000062,poisson,10,2,5,"{'criterion': 'poisson', 'max_depth': 10, 'min...",0.756142,0.601711,0.797936,0.683359,0.739847,0.715799,0.067854,117
131,0.005982,0.000270,0.002811,0.000177,poisson,10,2,10,"{'criterion': 'poisson', 'max_depth': 10, 'min...",0.792084,0.643696,0.806966,0.737417,0.745343,0.745101,0.057231,107
132,0.006049,0.000146,0.002955,0.000063,poisson,10,4,2,"{'criterion': 'poisson', 'max_depth': 10, 'min...",0.762739,0.708540,0.810793,0.762267,0.755166,0.759901,0.032447,96
133,0.005892,0.000092,0.002762,0.000063,poisson,10,4,5,"{'criterion': 'poisson', 'max_depth': 10, 'min...",0.762739,0.708540,0.810793,0.762267,0.755166,0.759901,0.032447,96


In [33]:
age_input = float(input("Age:"))
bmi_input = float(input("BMI:"))
children_input = float(input("Children:"))
sex_male_input = int(input("Sex male 0 or 1:"))
smoker_yes_input = int(input("smoker yes 0 or 1:"))

Age: 26
BMI: 30
Children: 0
Sex male 0 or 1: 1
smoker yes 0 or 1: 0


In [35]:
Future_Prediction = grid_search.predict([[age_input,bmi_input,children_input,sex_male_input,smoker_yes_input]])
print("Future_prediction={}".format(Future_Prediction))

Future_prediction=[15165.8017]
